In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:

import pandas as pd

# Load RBA Cash Rate
rba = pd.read_excel(
    "rba_F1_20261015_063926.xlsx",
    usecols="A,D",
    skiprows=11,
    header=None,
    names=["Date", "Cash_Rate_Percent"]
)

# Load ASX 200
asx = pd.read_csv("asx200_retrieved.csv")

# Display the first five records
print("RBA DATA:")
print(rba.head())

print("\nASX 200 DATA:")
print(asx.head())

# Check dataset sizes
print("\nRBA observations:", len(rba))
print("ASX observations:", len(asx))


RBA DATA:
        Date  Cash_Rate_Percent
0 2011-01-04               4.75
1 2011-01-05               4.75
2 2011-01-06               4.75
3 2011-01-07               4.75
4 2011-01-10               4.75

ASX 200 DATA:
                        Date         Open         High          Low  \
0  2026-06-16 00:00:00+10:00  8914.000000  8917.700195  8834.000000   
1  2026-06-17 00:00:00+10:00  8917.700195  8976.799805  8888.099609   
2  2026-06-18 00:00:00+10:00  8966.299805  8983.799805  8911.099609   
3  2026-06-19 00:00:00+10:00  8911.099609  8911.099609  8802.599609   
4  2026-06-22 00:00:00+10:00  8828.700195  8851.799805  8793.799805   

         Close    Adj Close   Volume  Dividends  Stock Splits  
0  8917.700195  8917.700195   724500        0.0           0.0  
1  8966.299805  8966.299805   694500        0.0           0.0  
2  8911.099609  8911.099609   803800        0.0           0.0  
3  8828.700195  8828.700195  1937000        0.0           0.0  
4  8816.099609  8816.099609   672200

In [2]:

# PART 2 - Data Quality Checking

print("=== RBA DATA QUALITY ===")

print("Missing Cash Rate:",
      rba["Cash_Rate_Percent"].isna().sum())

print("Duplicate Dates:",
      rba["Date"].duplicated().sum())

print("Missing Dates:",
      rba["Date"].isna().sum())


print("\n=== ASX 200 DATA QUALITY ===")

print("Missing Closing Prices:",
      asx["Close"].isna().sum())

print("Duplicate Dates:",
      asx["Date"].duplicated().sum())

print("Missing Dates:",
      asx["Date"].isna().sum())

print("Invalid Closing Prices:",
      (pd.to_numeric(asx["Close"], errors="coerce") <= 0).sum())


=== RBA DATA QUALITY ===
Missing Cash Rate: 1
Duplicate Dates: 0
Missing Dates: 0

=== ASX 200 DATA QUALITY ===
Missing Closing Prices: 0
Duplicate Dates: 0
Missing Dates: 0
Invalid Closing Prices: 0


In [3]:

# PART 2 - Date Standardisation

rba_clean = rba.copy()
asx_clean = asx.copy()

# Standardise RBA dates
rba_clean["Date"] = pd.to_datetime(
    rba_clean["Date"], errors="coerce"
).dt.date

# Convert ASX timestamps to Sydney dates
asx_clean["Date"] = (
    pd.to_datetime(
        asx_clean["Date"],
        utc=True,
        errors="coerce"
    )
    .dt.tz_convert("Australia/Sydney")
    .dt.date
)

# Identify the missing RBA cash rate
print("=== MISSING RBA RECORD ===")
print(
    rba_clean[
        rba_clean["Cash_Rate_Percent"].isna()
    ][["Date", "Cash_Rate_Percent"]]
)

# Match the available ASX date range
start_date = asx_clean["Date"].min()
end_date = asx_clean["Date"].max()

rba_filtered = rba_clean[
    rba_clean["Date"].between(start_date, end_date)
].copy()

print("\n=== DATE RANGE ===")
print("ASX Start:", start_date)
print("ASX End:", end_date)
print("RBA records in range:", len(rba_filtered))

print("\n=== ASX STANDARDISED DATES ===")
print(asx_clean[["Date", "Close"]].head())


=== MISSING RBA RECORD ===
            Date  Cash_Rate_Percent
3994  2026-10-14                NaN

=== DATE RANGE ===
ASX Start: 2026-06-16
ASX End: 2026-10-13
RBA records in range: 86

=== ASX STANDARDISED DATES ===
         Date        Close
0  2026-06-16  8917.700195
1  2026-06-17  8966.299805
2  2026-06-18  8911.099609
3  2026-06-19  8828.700195
4  2026-06-22  8816.099609


In [4]:

# PART 2 - Data Alignment

# Prepare the two datasets
asx_ready = asx_clean[["Date", "Close"]].copy()

rba_ready = rba_filtered[
    ["Date", "Cash_Rate_Percent"]
].copy()

# Ensure financial values are numeric
asx_ready["Close"] = pd.to_numeric(
    asx_ready["Close"], errors="coerce"
)

rba_ready["Cash_Rate_Percent"] = pd.to_numeric(
    rba_ready["Cash_Rate_Percent"], errors="coerce"
)

# Align datasets by Sydney calendar date
aligned_data = pd.merge(
    asx_ready,
    rba_ready,
    on="Date",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Rename columns for clarity
aligned_data = aligned_data.rename(columns={
    "Close": "ASX_Close",
    "Cash_Rate_Percent": "RBA_Rate"
})

# Check alignment results
print("=== DATA ALIGNMENT RESULTS ===")
print("Total ASX records:", len(asx_ready))
print("Total aligned records:", len(aligned_data))

print("Matching dates:",
      (aligned_data["_merge"] == "both").sum())

print("Unmatched ASX dates:",
      (aligned_data["_merge"] == "left_only").sum())

print("Missing RBA rates after alignment:",
      aligned_data["RBA_Rate"].isna().sum())

print("\n=== FIRST 10 RECORDS ===")
print(aligned_data.head(10))

print("\n=== UNMATCHED RECORDS ===")
print(
    aligned_data[
        (aligned_data["_merge"] != "both") |
        (aligned_data["RBA_Rate"].isna())
    ]
)


=== DATA ALIGNMENT RESULTS ===
Total ASX records: 86
Total aligned records: 86
Matching dates: 86
Unmatched ASX dates: 0
Missing RBA rates after alignment: 0

=== FIRST 10 RECORDS ===
         Date    ASX_Close  RBA_Rate _merge
0  2026-06-16  8917.700195      4.35   both
1  2026-06-17  8966.299805      4.35   both
2  2026-06-18  8911.099609      4.35   both
3  2026-06-19  8828.700195      4.35   both
4  2026-06-22  8816.099609      4.35   both
5  2026-06-23  8787.000000      4.35   both
6  2026-06-24  8808.400391      4.35   both
7  2026-06-25  8748.700195      4.35   both
8  2026-06-26  8764.200195      4.35   both
9  2026-06-29  8823.400391      4.35   both

=== UNMATCHED RECORDS ===
Empty DataFrame
Columns: [Date, ASX_Close, RBA_Rate, _merge]
Index: []


In [5]:

# PART 2 - Financial Metrics Calculation

# Prepare curated dataset
curated = aligned_data.drop(columns=["_merge"]).copy()

# Sort by trading date
curated = curated.sort_values("Date").reset_index(drop=True)

# Check closing prices before calculation
assert curated["ASX_Close"].notna().all()
assert (curated["ASX_Close"] > 0).all()

# Metric 1: 10-Day Moving Average
curated["MA_10"] = (
    curated["ASX_Close"]
    .rolling(window=10, min_periods=10)
    .mean()
)

# Metric 2: Daily Percentage Return
curated["Daily_Return"] = (
    curated["ASX_Close"]
    .pct_change(fill_method=None) * 100
)

# Metric 3: 14-Day Volatility
curated["Volatility_14"] = (
    curated["Daily_Return"]
    .rolling(window=14, min_periods=14)
    .std(ddof=1)
)

print("=== FINANCIAL METRICS RESULTS ===")
print(curated.tail(10).to_string(index=False))

print("\nTotal records:", len(curated))


=== FINANCIAL METRICS RESULTS ===
      Date   ASX_Close  RBA_Rate       MA_10  Daily_Return  Volatility_14
2026-09-30 8789.299805       4.6 8726.390039      0.918558       0.526586
2026-10-01 8614.400391       4.6 8714.590039     -1.989913       0.714702
2026-10-02 8682.099609       4.6 8709.679980      0.785884       0.752144
2026-10-05 8686.400391       4.6 8705.129980      0.049536       0.713110
2026-10-06 8735.700195       4.6 8702.920020      0.567552       0.725486
2026-10-07 8727.700195       4.6 8699.160059     -0.091578       0.717742
2026-10-08 8660.900391       4.6 8695.050098     -0.765377       0.746277
2026-10-09 8716.599609       4.6 8700.210059      0.643111       0.769357
2026-10-12 8783.200195       4.6 8710.560059      0.764066       0.793443
2026-10-13 8783.900391       4.6 8718.020117      0.007972       0.793247

Total records: 86


In [6]:

# PART 2 - Proof of Concept Validation

import numpy as np

# Verify Daily Percentage Return independently
previous_close = curated.loc[0, "ASX_Close"]
current_close = curated.loc[1, "ASX_Close"]

manual_return = (
    (current_close - previous_close)
    / previous_close * 100
)

python_return = curated.loc[1, "Daily_Return"]

print("=== DAILY RETURN VALIDATION ===")
print("Previous Close:", previous_close)
print("Current Close:", current_close)
print("Manual Calculation:", manual_return)
print("Python Calculation:", python_return)

assert np.isclose(manual_return, python_return)

print("Validation PASSED!")


=== DAILY RETURN VALIDATION ===
Previous Close: 8917.7001953125
Current Close: 8966.2998046875
Manual Calculation: 0.5449791797278171
Python Calculation: 0.5449791797278225
Validation PASSED!


In [7]:

# PART 2 - Additional Metric Validation

import numpy as np

# Validate 10-Day Moving Average
manual_ma10 = np.mean(
    curated["ASX_Close"].iloc[0:10]
)

python_ma10 = curated.loc[9, "MA_10"]

print("=== 10-DAY MOVING AVERAGE ===")
print("Manual:", manual_ma10)
print("Python:", python_ma10)

assert np.isclose(manual_ma10, python_ma10)

print("MA_10 Validation PASSED!")


# Validate 14-Day Volatility
manual_volatility = np.std(
    curated["Daily_Return"].iloc[1:15],
    ddof=1
)

python_volatility = curated.loc[14, "Volatility_14"]

print("\n=== 14-DAY VOLATILITY ===")
print("Manual:", manual_volatility)
print("Python:", python_volatility)

assert np.isclose(
    manual_volatility, python_volatility
)

print("Volatility Validation PASSED!")


=== 10-DAY MOVING AVERAGE ===
Manual: 8837.16005859375
Python: 8837.16005859375
MA_10 Validation PASSED!

=== 14-DAY VOLATILITY ===
Manual: 0.6309204991020105
Python: 0.6309204991020106
Volatility Validation PASSED!


In [8]:

# PART 2 - Export Curated Data and Quality Log

import os
import pandas as pd

os.makedirs("curated", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Export analysis-ready dataset
final_data = curated[
    ["Date", "ASX_Close", "RBA_Rate",
     "MA_10", "Daily_Return", "Volatility_14"]
].copy()

final_data["Date"] = pd.to_datetime(
    final_data["Date"]
).dt.strftime("%Y-%m-%d")

final_data.to_csv(
    "curated/curated_market_data.csv",
    index=False
)

# Record missing RBA observations
quality_log = rba_clean[
    rba_clean["Cash_Rate_Percent"].isna()
][["Date"]].copy()

quality_log["Source"] = "RBA"
quality_log["Issue"] = "Missing cash rate"
quality_log["Action"] = (
    "Logged; excluded from current analysis "
    "window; no value imputed"
)

quality_log.to_csv(
    "logs/part2_data_quality_log.csv",
    index=False
)

print("=== PART 2 EXPORT RESULTS ===")
print("Curated records:", len(final_data))
print("Quality issues logged:", len(quality_log))

print("\nMissing calculated metrics:")
print(final_data[
    ["MA_10", "Daily_Return", "Volatility_14"]
].isna().sum())

print("\nFiles created successfully!")
print("curated/curated_market_data.csv")
print("logs/part2_data_quality_log.csv")


=== PART 2 EXPORT RESULTS ===
Curated records: 86
Quality issues logged: 1

Missing calculated metrics:
MA_10             9
Daily_Return      1
Volatility_14    14
dtype: int64

Files created successfully!
curated/curated_market_data.csv
logs/part2_data_quality_log.csv
